# PDF Summarizer Agent — QLoRA fine-tune of Qwen2.5-7B-Instruct

**MANUAL_ACTION_REQUIRED before running:**
1. Open this notebook on Kaggle (GPU T4/P100, or Colab GPU runtime).
2. Attach a GPU accelerator.
3. Add secrets `HF_TOKEN` (write-scope Hugging Face token) via the platform's secrets UI — Kaggle: notebook editor Add-ons > Secrets; Colab: key icon in the left sidebar, and enable "Notebook access" for the secret or it stays invisible to the notebook even though it exists.
4. Run cells top to bottom. This is NOT executable end-to-end without a GPU session — Kaggle/Colab have no API-based execution, so this is always a manual run.

This notebook pulls the dataset from a private HF Hub dataset repo (pushed by `pdfsum publish-dataset` from the local control center) and checkpoints to a private HF Hub model repo after every save, so an interrupted free GPU session can resume rather than losing progress.

In [ ]:
# expandable_segments is most effective set before CUDA initializes
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers peft bitsandbytes accelerate datasets huggingface_hub

In [ ]:
import json

import torch
from huggingface_hub import HfApi, hf_hub_download, login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

# Try local env var first, then platform-specific secret stores, so this
# notebook works unmodified on Kaggle or Colab.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            pass
assert hf_token, "MANUAL_ACTION_REQUIRED: set the HF_TOKEN secret (see markdown cell above)"
login(token=hf_token)

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # apache-2.0, verified 2026-08-18 — do not
                                          # swap for a different Qwen2.5 size without
                                          # re-checking ITS specific license page
DATASET_REPO = "makremmakrem/pdf-summarizer-agent-dataset"
MODEL_REPO = "makremmakrem/pdf-summarizer-agent-qlora"
OUTPUT_DIR = "./checkpoints"

dataset_path = hf_hub_download(
    repo_id=DATASET_REPO, filename="examples.jsonl", repo_type="dataset", token=hf_token
)
with open(dataset_path) as f:
    examples = [json.loads(line) for line in f]

before = len(examples)
examples = [e for e in examples if e.get("length_compliant")]
print(f"{len(examples)}/{before} examples kept (length_compliant only — see project memory: "
      "training on non-conforming length examples teaches the wrong lesson)")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

IGNORE_INDEX = -100
# Free-tier 16GB GPUs (Kaggle P100 / Colab T4) can't realistically train on
# tens-of-thousands-of-token sequences even in 4-bit -- this is a hardware
# budget, not a guess about the data (measuring real lengths still matters,
# see below -- it tells us how much truncation we're actually doing).
MAX_SEQ_LENGTH = 4096


def build_prompt(document_text, domain):
    return (
        f"Summarize this {domain} document. Respond with ONLY a JSON "
        f"object -- no prose, no markdown fences.\n\nDOCUMENT:\n{document_text}"
    )


def build_labels(document_text, domain, assistant_content, max_len=MAX_SEQ_LENGTH):
    """Truncates the INPUT document text (never the assistant response) to
    fit max_len via binary search on character count. Truncating the final
    token sequence from the end instead -- the naive approach -- would cut
    into the label region for exactly the long documents (10-Ks, book
    excerpts) meant to demonstrate the long-document-handling target,
    silently corrupting or dropping the response entirely. Verified
    2026-08-18: Qwen2.5's chat template has no {% generation %} block, so
    return_assistant_tokens_mask silently returns an all-zero mask -- this
    prefix-tokenization workaround was confirmed to be a true prefix for
    this template; re-verify if the base model changes."""
    lo, hi = 0, len(document_text)
    best_full_ids, best_prefix_ids, best_cut = None, None, None
    while lo <= hi:
        mid = (lo + hi) // 2
        candidate_prompt = build_prompt(document_text[:mid], domain)
        prefix_ids = tokenizer.apply_chat_template(
            [{"role": "user", "content": candidate_prompt}],
            tokenize=True, add_generation_prompt=True,
        )["input_ids"]
        full_ids = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": candidate_prompt},
                {"role": "assistant", "content": assistant_content},
            ],
            tokenize=True, add_generation_prompt=False,
        )["input_ids"]
        if len(full_ids) <= max_len:
            best_full_ids, best_prefix_ids, best_cut = full_ids, prefix_ids, mid
            lo = mid + 1
        else:
            hi = mid - 1

    if best_full_ids is None:
        # Even an empty document + the assistant response doesn't fit --
        # the response itself is too long for max_len. Not recoverable here.
        return None, None, False

    assert best_full_ids[: len(best_prefix_ids)] == best_prefix_ids, (
        "prompt-only tokenization is not a true prefix -- masking workaround unsafe here"
    )
    labels = [IGNORE_INDEX] * len(best_prefix_ids) + list(best_full_ids[len(best_prefix_ids):])
    was_truncated = best_cut < len(document_text)
    return best_full_ids, labels, was_truncated


tokenized = []
truncated_count = 0
skipped_count = 0
for ex in examples:
    input_ids, labels, was_truncated = build_labels(
        ex["document_text"], ex["domain"], ex["teacher_output_raw"]
    )
    if input_ids is None:
        skipped_count += 1
        continue
    truncated_count += was_truncated
    tokenized.append({"input_ids": input_ids, "labels": labels})

print(f"{len(tokenized)} examples tokenized, {truncated_count} had their input document "
      f"truncated to fit MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}, {skipped_count} skipped "
      "(response alone exceeded MAX_SEQ_LENGTH)")

lengths = [len(t["input_ids"]) for t in tokenized]
print(f"token length: min={min(lengths)} p50={sorted(lengths)[len(lengths)//2]} "
      f"max={max(lengths)}")


In [ ]:
def pad_to_length(example, max_len, pad_id):
    input_ids = example["input_ids"]
    labels = example["labels"]
    pad_len = max_len - len(input_ids)
    attention_mask = [1] * len(input_ids) + [0] * pad_len
    input_ids = input_ids + [pad_id] * pad_len
    labels = labels + [IGNORE_INDEX] * pad_len
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
processed = [pad_to_length(t, MAX_SEQ_LENGTH, pad_id) for t in tokenized]

split = max(1, int(len(processed) * 0.9))
train_data, eval_data = processed[:split], processed[split:]
print(f"train: {len(train_data)}  eval: {len(eval_data)}")


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return {k: torch.tensor(v) for k, v in self.items[i].items()}


train_dataset = ListDataset(train_data)
eval_dataset = ListDataset(eval_data) if eval_data else None


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# per_device_eval_batch_size set explicitly — it defaults to 8 even with
# per_device_train_batch_size=1, and can OOM during eval from float32 logit
# upcasting on a multi-example batch (lesson from the sibling project).
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="steps",
    save_steps=25,
    eval_strategy="steps" if eval_dataset else "no",
    eval_steps=25,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    bf16=True,
    report_to="none",
    push_to_hub=True,
    hub_model_id=MODEL_REPO,
    hub_private_repo=True,
    hub_strategy="every_save",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Checkpoint-upload-and-resume against HF Hub after every save is what makes
# a free, interruptible GPU session viable — if this cell's session dies,
# re-run with resume_from_checkpoint=True on the next free session.
RESUME = False  # set True after an interruption
trainer.train(resume_from_checkpoint=RESUME)

In [ ]:
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub()
print(f"pushed final adapter to https://huggingface.co/{MODEL_REPO}")